In [4]:
import numpy as np
import pickle
import os

In [9]:
# I will use mc_events_chunk02 as the and the seeds are 01 to 10 data chunks
print("Loading MC scalers...")
mc_scaler_path = "SemiVisJets/mc_scalers"
mc_scaler = {}
for i in range(1, 6):
    with open(f"{mc_scaler_path}/mc_scaler_chunk{i:02d}.pkl","rb") as f:
        print("Loading trained minmax scaler.")
        scaler = pickle.load(f)
    mc_scaler[i] = scaler

# Pick a scaler to use
seed = 2
MC_scale = mc_scaler[seed]
MC_scale_test = mc_scaler[1]  # Use chunk 01 scaler for test set

Loading MC scalers...
Loading trained minmax scaler.
Loading trained minmax scaler.
Loading trained minmax scaler.
Loading trained minmax scaler.
Loading trained minmax scaler.


In [10]:
print("Loading data... Chunk 01 - 10")

var_names = ["ht", "met", "m_jj", "tau21_j1", "tau21_j2", "tau32_j1", "tau32_j2"]
data_path = "data/chunks"
research_proj = "SemiVisJets"

data_events = {}
for i in range(1, 11):
    data = np.load(f"{data_path}/data_chunk_{i:02d}.npz", allow_pickle=True)
    events = np.column_stack([data[var] for var in var_names])
    data_events[i] = events
    print(f"Chunk {i:02d} shape: {events.shape}")



Loading data... Chunk 01 - 10
Chunk 01 shape: (10000000, 7)
Chunk 02 shape: (10000000, 7)
Chunk 03 shape: (10000000, 7)
Chunk 04 shape: (10000000, 7)
Chunk 05 shape: (10000000, 7)
Chunk 06 shape: (10000000, 7)
Chunk 07 shape: (10000000, 7)
Chunk 08 shape: (10000000, 7)
Chunk 09 shape: (10000000, 7)
Chunk 10 shape: (10000000, 7)


In [11]:
def sr_mask(events):
    ht = events[:,0]
    met = events[:,1]
    # Apply SR cuts
    mask = (ht > 600) & (met > 600)
    return mask

data_mask_SR_test = sr_mask(data_events[6])  # Using chunk 06 as test set
data_mask_CR = ~data_mask_SR_test
print(f"Test Chunk 06 total events: {data_events[6].shape}")
print(f"Test Chunk 06: SR Events: {np.sum(data_mask_SR_test)}, CR Events: {np.sum(data_mask_CR)}")
os.makedirs(f"{research_proj}/data/data_test", exist_ok=True)
np.savez(f"{research_proj}/data/data_test/data_events_chunk06.npz", data_events_cr=MC_scale_test.transform(data_events[6][data_mask_CR]), data_events_sr=MC_scale_test.transform(data_events[6][data_mask_SR_test]))    

# for i in range(1, 6):
#     data_mask_SR = sr_mask(data_events[i])
#     data_mask_CR = ~data_mask_SR
#     print(f"Chunk {i:02d} total events: {data_events[i].shape}")
#     print(f"Chunk {i:02d}: SR Events: {np.sum(data_mask_SR)}, CR Events: {np.sum(data_mask_CR)}")
#     os.makedirs(f"{research_proj}/data/data_seed{seed}", exist_ok=True)
#     np.savez(f"{research_proj}/data/data_seed{seed}/data_events_chunk{i:02d}.npz", data_events_cr=MC_scale.transform(data_events[i][data_mask_CR]), data_events_sr=MC_scale.transform(data_events[i][data_mask_SR]))
        

Test Chunk 06 total events: (10000000, 7)
Test Chunk 06: SR Events: 16365, CR Events: 9983635
